# 04 · Ejercicios extra (backup)

**Tiempo estimado:** 30-45 min (si quedan ganas tras 01-03).

**Objetivos.** Tocar técnicas que las sesiones de teoría mencionan pero que los notebooks 01-03 no practican, o que son clásicas en hidrología y aún no han aparecido:

1. Heatmap calendario (mes × año) del caudal.
2. Rachas de caudal bajo (sequías hidrológicas con `cumsum`).
3. Transformaciones Box-Cox vs Yeo-Johnson sobre lluvia diaria (con ceros).
4. Test de Ljung-Box sobre los residuos de STL.
5. Periodograma para descubrir periodicidades.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

caudal = ud.cargar_caudal_genil()
lluvia_d = ud.cargar_lluvia_genil()

## 1 · Heatmap calendario mes × año

El boxplot por mes (notebook 01) muestra la **estacionalidad** pero promedia entre años. El heatmap mes×año muestra **simultáneamente** estacionalidad (columnas) y años atípicos (filas) — sequías y años húmedos aparecen como bandas horizontales.


In [ ]:
df = caudal.loc["1975":"2020"].dropna().to_frame("Q")
df["anio"], df["mes"] = df.index.year, df.index.month
mat = df.groupby(["anio", "mes"])["Q"].mean().unstack("mes")

fig, ax = plt.subplots(figsize=(7, 8))
im = ax.imshow(
    np.log10(mat.values),
    aspect="auto",
    cmap="viridis",
    extent=[0.5, 12.5, mat.index.max() + 0.5, mat.index.min() - 0.5],
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(list("EFMAMJJASOND"))
ax.set_ylabel("Año")
ax.set_title("Caudal medio mensual del Genil — log10(m³/s)")
plt.colorbar(im, ax=ax, label="log10 m³/s")
plt.tight_layout()

**Lectura:**

- La sequía 2004-08 aparece como banda horizontal oscura.
- Los picos invernales son columnas verticales (E-F-M).
- La escala log es necesaria por la fuerte asimetría de Q (sin ella el mapa quedaría aplastado en los valores bajos).

**Variantes para probar:**

- Cambiar a `mat = ... .sum().unstack("mes")` sobre lluvia y comparar.
- Sustituir la escala log por una normalización por columnas (anomalía respecto a la media mensual) para ver años atípicos relativos al ciclo estacional.


## 2 · Rachas de caudal bajo (sequías hidrológicas)

Idioma idiomático de pandas para detectar rachas: `(s != s.shift()).cumsum()` etiqueta cada "racha" de valores iguales con un id distinto. Combinándolo con `groupby` se identifican las **N rachas más largas** bajo un umbral.

Definimos "día seco hidrológico" como aquel con caudal por debajo del percentil 10 del periodo de referencia.


In [ ]:
q = caudal.loc["1995":"2020"].dropna()
umbral = q.quantile(0.10)
print(f"Umbral P10: {umbral:.2f} m³/s")

seco = (q < umbral).astype(int)
grupos = (seco != seco.shift()).cumsum()
rachas = seco.groupby(grupos).sum()
rachas = rachas[rachas > 0].sort_values(ascending=False)

print(f"\nNº de rachas secas: {len(rachas)}")
print(f"Racha mediana    : {rachas.median():.0f} días")
print(f"Racha más larga  : {rachas.max():.0f} días")
print("\nTop 5 rachas secas:")
print(rachas.head())

In [ ]:
# ¿Cuándo ocurrió la racha más larga?
grupo_max = rachas.idxmax()
mascara = grupos == grupo_max
fechas = q.index[mascara]
print(f"Racha más larga: {fechas[0].date()} → {fechas[-1].date()}  ({len(fechas)} días)")

# Visualización del periodo
ventana = q.loc[fechas[0] - pd.Timedelta(days=30) : fechas[-1] + pd.Timedelta(days=30)]
fig, ax = plt.subplots()
ax.plot(ventana.index, ventana.values, color="#1f6f8b", lw=0.6)
ax.axhline(umbral, color="red", ls="--", lw=0.8, label=f"P10 = {umbral:.1f} m³/s")
ax.axvspan(fechas[0], fechas[-1], color="#fbbf24", alpha=0.3, label="racha")
ax.set_ylabel("Q (m³/s)")
ax.legend()
plt.tight_layout()

**Discusión:** este patrón `(s != s.shift()).cumsum()` reaparece constantemente — para detectar gaps, eventos por encima de un umbral, periodos de funcionamiento de un sensor… vale la pena interiorizarlo.

El "índice de sequía" más simple posible es contar días por año bajo el P10. Variantes profesionales:

- **SPI** (*Standardized Precipitation Index*): igual pero sobre lluvia acumulada en una ventana móvil de N meses, normalizado.
- **SDI** (*Streamflow Drought Index*): SPI aplicado a caudal.


## 3 · Transformaciones: log, Box-Cox y Yeo-Johnson

Las slides introducen Box-Cox (necesita $y>0$) y Yeo-Johnson (admite ceros y negativos). La lluvia diaria es el caso de uso ideal de Yeo-Johnson: distribución muy asimétrica **con muchos ceros**.

Comparamos tres transformaciones que pretenden "normalizar" la distribución:

- $\log(1+P)$: clásica, simple, asintótica.
- Box-Cox con $\lambda$ óptimo (requiere clip de los ceros).
- Yeo-Johnson con $\lambda$ óptimo (admite ceros).


In [ ]:
from scipy.stats import boxcox, yeojohnson

lluvia = ud.cargar_lluvia_genil().dropna().values
print(f"n = {len(lluvia)}   ceros = {(lluvia == 0).mean():.1%}   max = {lluvia.max():.1f} mm")

y_log = np.log1p(lluvia)
y_bc, lam_bc = boxcox(np.clip(lluvia, 0.01, None))
y_yj, lam_yj = yeojohnson(lluvia)

print(f"\nλ Box-Cox    : {lam_bc:+.3f}")
print(f"λ Yeo-Johnson: {lam_yj:+.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.0))
axes[0].hist(lluvia, bins=60, color="#1f6f8b")
axes[0].set_title(f"Original (mm)\nskew={pd.Series(lluvia).skew():.1f}")
axes[1].hist(y_log, bins=60, color="#7c3aed")
axes[1].set_title(f"log(1+P)\nskew={pd.Series(y_log).skew():.1f}")
axes[2].hist(y_bc, bins=60, color="#c2410c")
axes[2].set_title(f"Box-Cox λ={lam_bc:.2f}\nskew={pd.Series(y_bc).skew():.1f}")
axes[3].hist(y_yj, bins=60, color="#16a34a")
axes[3].set_title(f"Yeo-Johnson λ={lam_yj:.2f}\nskew={pd.Series(y_yj).skew():.1f}")
for ax in axes:
    ax.set_yscale("log")
plt.tight_layout()

**Lectura:**

- La distribución original tiene un atomo en 0 (días sin lluvia) y una cola muy pesada.
- $\log(1+P)$ comprime la cola pero deja un pico bestial en 0.
- Box-Cox con clip introduce un sesgo artificial: todos los ceros se mapean al mismo valor "casi -∞".
- Yeo-Johnson maneja el 0 de forma natural y suele dar la skewness más baja.

**¿Por qué importa?** Modelos como ARIMA o regresiones asumen residuos gaussianos. Una transformación previa cambia mucho el comportamiento del modelo (lo veremos en sesión 2). Para variables 0-infladas, **Yeo-Johnson** es la opción honesta.

> *Aviso:* "skewness baja" no equivale a "Gaussiana". Ninguna de estas transformaciones convierte una distribución 0-inflada en normal — solo simétrizan la cola. Para predicción de eventos extremos a menudo es preferible modelar **ocurrencia** (P>0) y **magnitud** (P|P>0) por separado.


## 4 · Test de Ljung-Box sobre residuos STL

Las slides definen "ruido blanco" como el objetivo del residuo tras descomponer/modelar, y mencionan Ljung-Box como el test formal. Vamos a aplicarlo:

$$
Q(h) = n(n+2) \sum_{k=1}^{h} \frac{\hat\rho_k^2}{n-k} \;\sim\; \chi^2_h
$$

bajo $H_0$: los primeros $h$ lags de la ACF son cero (la serie es ruido blanco).


In [ ]:
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox

caudal_mensual = caudal.resample("MS").mean().loc["1995":"2020"].interpolate("linear", limit=2)
stl = STL(caudal_mensual.dropna(), period=12, robust=True).fit()
resid_stl = stl.resid.dropna()

lb_stl = acorr_ljungbox(resid_stl, lags=[6, 12, 24], return_df=True)
print("Ljung-Box sobre residuos STL:")
print(lb_stl)
print(f"\n¿Es ruido blanco? (todos p > 0.05): {(lb_stl['lb_pvalue'] > 0.05).all()}")

In [ ]:
# Comparativa: residuos de un AR(1) ingenuo sobre el caudal mensual
phi = caudal_mensual.autocorr(lag=1)
resid_ar1 = (caudal_mensual - phi * caudal_mensual.shift(1)).dropna()
print(f"AR(1) ingenuo: φ = {phi:.3f}")

lb_ar1 = acorr_ljungbox(resid_ar1, lags=[6, 12, 24], return_df=True)
print("\nLjung-Box sobre residuos AR(1):")
print(lb_ar1)

# ACF visual para confirmar
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
plot_acf(resid_stl, lags=24, ax=axes[0])
axes[0].set_title("Residuos STL")
plot_acf(resid_ar1, lags=24, ax=axes[1])
axes[1].set_title("Residuos AR(1) ingenuo")
plt.tight_layout()

**Discusión:**

- Lo habitual es que **ni STL ni AR(1)** dejen ruido blanco sobre el caudal mensual: STL captura ciclo y tendencia pero deja la **persistencia mensual**, y AR(1) captura la persistencia pero deja la **estacionalidad**.
- La conclusión natural es combinar ambos: modelar el residuo STL con un AR, o (mejor) usar SARIMA, que mete los dos ingredientes en un único modelo. **Eso es exactamente lo que veremos en sesión 2.**
- Ljung-Box es la herramienta estándar para validar un ARIMA/SARIMA: si el residuo del modelo pasa el test, no hay (mucho) más zumo que extraer con un modelo lineal.

> *Detalle técnico:* cuando aplicas Ljung-Box a residuos de un modelo con $p+q$ parámetros, conviene pasar `model_df=p+q` para corregir los grados de libertad. Para residuos de STL no hace falta.


## 5 · Periodograma — descubrir ciclos sin asumirlos

La ACF nos confirma un ciclo cuando ya sospechamos que está ahí (vimos picos cada 365 días). El **periodograma** (transformada de Fourier al cuadrado) los **descubre** sin pista previa: muestra cuánta energía hay a cada frecuencia.

$$
P(f) = \frac{1}{n}\left|\sum_{t=1}^{n} y_t \, e^{-2\pi i f t}\right|^2
$$

Picos en $P(f)$ → periodicidades dominantes en $T = 1/f$.


In [ ]:
from scipy.signal import periodogram

s = caudal.loc["1995":"2020"].dropna()
f, Pxx = periodogram(s.values, fs=1.0)  # fs en 1/día
periodos = 1 / f[1:]  # excluimos f=0 (componente DC)

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.semilogy(periodos, Pxx[1:], color="#1f6f8b", lw=0.8)
ax.set_xlim(2, 800)
ax.set_xlabel("Periodo (días, escala lineal)")
ax.set_ylabel("PSD (escala log)")
ax.axvline(365, color="red", ls="--", lw=0.8, label="1 año")
ax.axvline(365 / 2, color="orange", ls="--", lw=0.8, label="6 meses")
ax.axvline(7, color="grey", ls=":", lw=0.8, label="semanal")
ax.legend()
ax.set_title("Periodograma del caudal diario del Genil (1995-2020)")
plt.tight_layout()

In [ ]:
# Top 5 picos del periodograma (excluyendo periodos > 5 años)
mascara = (periodos > 2) & (periodos < 365 * 5)
indices = np.argsort(Pxx[1:][mascara])[::-1][:10]
periodos_top = periodos[mascara][indices]
print("Periodos con mayor energía (días):")
for p in periodos_top:
    print(f"  {p:7.1f} días  ≈ {p / 30.4:.1f} meses")

**Lectura:**

- El pico anual ($T=365$) debe dominar — es la estacionalidad hidrológica del Genil.
- ¿Aparece un pico secundario en $T \approx 182$ (semestral)? Sería el segundo armónico del ciclo anual: indica que la estacionalidad no es senoidal sino con dos picos por año (lluvia + deshielo).
- ¿Hay algo cerca de $T=7$ días? Sería una **firma humana**: regulación semanal del embalse, riegos coordinados…
- Las frecuencias muy bajas (periodos > 5 años) están dominadas por la tendencia, no por ciclos reales.

**Cuándo usar periodograma vs ACF:**

- ACF: confirmar un ciclo conocido, medir su amortiguamiento, identificar órdenes ARIMA.
- Periodograma: **descubrir** periodos no anticipados (ciclos antrópicos, lunares, etc.).

> *Detalle:* el periodograma crudo es muy ruidoso. En análisis serios se usa **Welch** (`scipy.signal.welch`) que promedia ventanas → estimaciones más estables a costa de resolución en frecuencia.
